In [ ]:
#enable autoreload

%load_ext autoreload
%autoreload all

#common jupyter functions
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

import sys, os.path
importPath = os.path.abspath('../../common')
if not importPath in sys.path:
    sys.path.append(importPath)

In [ ]:
#define training logs path
trainingLogsPath = f"../logs/potion_mtclmtlbl_words_v2/"
print(trainingLogsPath)

#define path for data files
baseDataPath = "./data"
print(baseDataPath)

#define sequence length for the model (this is also used in tokenizer)
modelSeqLen = 1024

In [ ]:
#create helpers
import torch
from nns.utils import LitUtils

tdev = torch.device('cuda') 
lu = LitUtils(trainingLogsPath)

#load test dataset
from corpus.hdf5Dset import MultilabelDataset, MultilabelTorchDataset, MultilabelTorchDataLoader
dsetTest = MultilabelDataset.loadFromFiles(os.path.join(baseDataPath, "dataset-words-test"))

#evaluate all model checkpoints against the test set
from nns.potion_multiclass_multilabel_v2 import MultiClassMultiLabelTokenLabelerLit

versWCpoints = lu.getVersionsWithCheckpoints()

colModelVersion = []
colSampleIdx = []
colExpectedLabels = []
colFoundLabels = []
colConfidence = []

for modelVer in tqdm(versWCpoints, "Evaluating models"):
	#load model
	cpointPath = lu.getVersionCheckpointPath(modelVer)

	litModel = MultiClassMultiLabelTokenLabelerLit.load_from_checkpoint(cpointPath)
	model = litModel.model
	del litModel	
	
	model.to(tdev)
	model.eval()
	model.compile()

	#create the dataset
	torchDset = MultilabelTorchDataset(dset=dsetTest)
	torchDset.dev = tdev

	#run model over test samples
	stance = torch.compiler.set_stance("force_eager")

	try:
		for sampleIdx in tqdm(range(len(torchDset)), "Running model on test set"):
			#get sample data
			[[inputTknsOrEmbeds, inputTknsMask, tknIdx], tknLbls] = torchDset[sampleIdx]

			#add batch dimension to input tensors
			inputTknsOrEmbeds = inputTknsOrEmbeds.unsqueeze(dim = 0)
			inputTknsMask = inputTknsMask.unsqueeze(dim = 0)

			#run model against sample
			with torch.inference_mode():			
				resModel = model(inputTknIdsOrEmbeds=inputTknsOrEmbeds, inputTknMask=inputTknsMask) #shape=(batchLen, seqLen, numLabels, 2)

			#extract confidence vectors for the labels of given tokens
			resTknLblConfs = resModel[range(resModel.shape[0]), tknIdx, :] #shape=(batchLen, sum(self.numLabels))

			#compute label predictions in each head
			resTknLbls = torch.zeros_like(resTknLblConfs)
			headOffset = 0
			for numHeadLbls in model.numLabels:
				#get indices of predicted labels for the current head, shape=(batchLen, 1)
				predictedLbl = resTknLblConfs[range(resModel.shape[0]), headOffset:(headOffset + numHeadLbls)].argmax(dim=1)
				#offset indices back to the final vector space
				predictedLbl += headOffset
				#assign the predicted label
				resTknLbls[range(resModel.shape[0]), predictedLbl] = 1
				#move offset past current head
				headOffset += numHeadLbls

			#drop batch dimensinos
			resTknLblConfs = resTknLblConfs.squeeze().to(torch.device('cpu'))
			resTknLbls = resTknLbls.squeeze().to(torch.device('cpu'))

			#add to results
			colModelVersion.append(modelVer)
			colSampleIdx.append(sampleIdx)
			colExpectedLabels.append(tknLbls.numpy(force=True))
			colFoundLabels.append(resTknLbls.numpy(force=True))
			colConfidence.append(resTknLblConfs.numpy(force=True))
		
		#clean up
		del model
	finally:
		torch.compiler.set_stance("default")

#save the result
import polars as pl

df = pl.DataFrame({
	"model_version" : colModelVersion,
	"sample_idx" : colSampleIdx,
	"expected_labels": colExpectedLabels,
	"found_labels" :colFoundLabels,
	"confidence": colConfidence,
})
df.write_parquet(os.path.join(baseDataPath, "eval-on-words.parquet"))

In [ ]:
import polars as pl
df = pl.read_parquet(os.path.join(baseDataPath, "eval-on-words.parquet"))
df.head()